In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-03-20 14:39:13.784898


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii
Task: 14_loss_forecasting
Subtask: 02_pull_tsp_data


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [6]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('with tbl_base as (\n'
 '    select\n'
 '\t\tbigAccountId,\n'
 '\t\tFundingMonth,\n'
 '\t\tintOpenBKType,\n'
 '\t\tAmtFinanced,\n'
 '\t\tBookValue,\n'
 '\t\tMonthsOnBooks,\n'
 '\t\tDealerType,\n'
 '        ROW_NUMBER() OVER (PARTITION BY bigAccountId ORDER BY FundingMonth) '
 'AS row_num\n'
 '    FROM riskdb.analytics.tbltempstaticpool\n'
 ')\n'
 'select *\n'
 'from tbl_base\n'
 'where row_num = 1;')


### Write into df

In [7]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# show
df

Wall time: 5.14 s


,bigAccountId,FundingMonth,intOpenBKType,AmtFinanced,BookValue,MonthsOnBooks,DealerType,row_num
0,3,2000-06-01,NaN,24066.59,23377.9,20,None,1
1,10,1998-10-01,NaN,11214.36,7525.0,20,Franchise,1
2,44,1996-01-01,NaN,13031.38,11725.0,20,None,1
3,69,2002-01-01,NaN,18521.08,12725.0,20,None,1
4,119,1998-10-01,NaN,7377.36,7350.0,20,Franchise,1
...,...,...,...,...,...,...,...,...
414066,7665722,2024-03-01,13.0,39368.73,30925.0,20,Franchise,1
414067,7665939,2024-03-01,NaN,20352.08,15625.0,20,Franchise,1
414068,7668528,2024-03-01,7.0,22191.53,17816.0,20,Independent,1
414069,7668983,2024-03-01,NaN,17772.71,13525.0,20,Independent,1


### Convert funding month to datetime

In [8]:
df['FundingMonth'] = pd.to_datetime(df['FundingMonth'])
df['FundingMonth_first'] = df['FundingMonth'].dt.to_period('M').dt.to_timestamp()
df['loan_to_value'] = df['AmtFinanced'] / df['BookValue']
df

,bigAccountId,FundingMonth,intOpenBKType,AmtFinanced,BookValue,MonthsOnBooks,DealerType,row_num,FundingMonth_first,loan_to_value
0,3,2000-06-01,NaN,24066.59,23377.9,20,None,1,2000-06-01,1.029459
1,10,1998-10-01,NaN,11214.36,7525.0,20,Franchise,1,1998-10-01,1.490280
2,44,1996-01-01,NaN,13031.38,11725.0,20,None,1,1996-01-01,1.111418
3,69,2002-01-01,NaN,18521.08,12725.0,20,None,1,2002-01-01,1.455488
4,119,1998-10-01,NaN,7377.36,7350.0,20,Franchise,1,1998-10-01,1.003722
...,...,...,...,...,...,...,...,...,...,...
414066,7665722,2024-03-01,13.0,39368.73,30925.0,20,Franchise,1,2024-03-01,1.273039
414067,7665939,2024-03-01,NaN,20352.08,15625.0,20,Franchise,1,2024-03-01,1.302533
414068,7668528,2024-03-01,7.0,22191.53,17816.0,20,Independent,1,2024-03-01,1.245596
414069,7668983,2024-03-01,NaN,17772.71,13525.0,20,Independent,1,2024-03-01,1.314064


### Save as parquet

In [9]:
%%time

# save
str_filename = 'df_accounts.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

Wall time: 2.54 s


### Upload to s3

In [10]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 1.62 s


### Clean-up

In [11]:
os.remove(str_local_path)